# APM_50441_EP - Machine Learning 2 - Lab 3: Trees, ensembles, SVM and tabular deep learning

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/APM-50441-EP/APM_50441_EP-2026-2027/blob/main/PC3/APM_50441_EP_Lab_3_simplified.ipynb) or run it on your own computer (see *Working environment* below).

This lab illustrates the third lecture (*Trees, Ensemble Methods, SVM and Tabular Deep Learning*). Its goals are:
- to grow and prune a classification tree (CART),
- to understand why averaging trees works (bagging, random forests, extra trees) and to use the out-of-bag estimate,
- to implement AdaBoost and to tune a gradient boosting (learning rate, number of trees, early stopping),
- to use a support vector machine with a kernel,
- to compare all these methods honestly, with the cost-based gain of Lab 1.

We use `Python` with `scikit-learn`, `polars` and `seaborn`.

**Working environment.** Colab is the simplest option: open the notebook of this lab from the course repository (*Open in Colab* link) and run it. To work on your own computer, download the lab folder (the notebook and its files) or clone the course repository, and open the notebook in an IDE that runs Jupyter notebooks (VS Code with the Python and Jupyter extensions, JupyterLab, Spyder, PyCharm). Create a Python environment once with [`uv`](https://docs.astral.sh/uv/): `uv venv`, activate it (`source .venv/bin/activate`, or `.venv\Scripts\activate` on Windows) and `uv pip install jupyter`, or build the whole course environment with `uv sync` in `PC/env`; then select this environment as the kernel of the notebook. The installation cell below installs the packages of the lab into the Python running the notebook (Colab or your environment), and the data are read from the local files when they are present (otherwise they are downloaded).

**Use of AI.** AI assistants are allowed, but you must be able to explain every line you use, and you should answer the *predict before you run* questions before running the code: the suggested code may not be correct!

All the libraries of the core part are installed at once. On Colab, restart the session once after this installation; locally, restart the kernel if the installation upgraded a package that was already loaded.

In [ ]:
# Installs into the Python running this notebook: Colab, or locally an activated virtual environment
import sys
!uv pip install --python "{sys.executable}" "scikit-learn==1.9.1" "polars==1.44.2" "seaborn==0.13.2" "fairlearn==0.14.0"

We use again the German credit data of Lab 1 and its cost model (`apm_credit.py`): the scorers `cost_scorer` and `cost_scorer_from_proba` return the real-world *gain* (the opposite of the cost) of the predictions, and of the Bayes decision computed from the probabilities.

In [ ]:
import os
import urllib.request

# each file is downloaded only if there is no local copy
CSV_URL = "https://raw.githubusercontent.com/IBM/monitor-wml-model-with-watson-openscale/master/data/german_credit_data_biased_training.csv"
APM_CREDIT_URL = "https://raw.githubusercontent.com/APM-50441-EP/APM_50441_EP-2026-2027/main/common/apm_credit.py"  # course repository
for url in [CSV_URL, APM_CREDIT_URL]:
  filename = url.split('/')[-1]
  if not os.path.exists(filename):
    urllib.request.urlretrieve(url, filename)

The data, the split, the preprocessing and the cost model are given (Lab 1).

In [ ]:
import time
import numpy as np
import polars as pl
import polars.selectors as cs
import seaborn.objects as so
import matplotlib.pyplot as plt
import sklearn
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, RepeatedStratifiedKFold
from threadpoolctl import threadpool_limits
threadpool_limits(limits=2, user_api='openmp')  # on such small datasets, many threads are slower than a few

import apm_credit
from apm_credit import cost_scorer, cost_scorer_from_proba, cost_metric

german_credit = pl.read_csv("german_credit_data_biased_training.csv")
X = german_credit.drop('Risk')
y = german_credit['Risk']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
LoanAmount_train, LoanAmount_test = X_train['LoanAmount'], X_test['LoanAmount']
apm_credit.set_proportions(y_train, {'Risk': .02, 'No Risk': .98})

categorical_features = X.select(cs.string()).columns
numerical_features = X.select(cs.numeric()).columns
preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
    ('num', StandardScaler(), numerical_features)]).set_output(transform='polars')

def make_pipeline(classifier):
  return Pipeline(steps=[('preprocessor', clone(preprocessor)), ('classifier', classifier)])

# the preprocessed matrices, for the methods implemented by hand
Xp_train = clone(preprocessor).fit(X_train).transform(X_train)
Xp_test = clone(preprocessor).fit(X_train).transform(X_test)
feature_names = Xp_train.columns
Xp_train, Xp_test = Xp_train.to_numpy(), Xp_test.to_numpy()
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
Xp_train.shape

## 1. Classification trees

CART grows a tree greedily: each split is chosen to decrease an impurity (Gini index by default) as much as possible, until the leaves are pure or too small. A fully grown tree overfits; it is then *pruned* along the sequence of subtrees minimizing the penalized criterion $\text{error} + \alpha \times \text{number of leaves}$ (cost-complexity pruning), $\alpha$ being chosen by cross validation.

**Question** *Predict before you run:* what are the training and test accuracies of a fully grown tree? Fit it, and report its depth and its number of leaves.

**Answer**

---

**Question** Compute the cost-complexity pruning path of the full tree (`cost_complexity_pruning_path`). For about $30$ values of $\alpha$ along the path, compute the training accuracy and the $5$-fold CV accuracy, and plot them against the number of leaves. Select $\alpha$ by cross validation, and plot the top of the pruned tree.

**Answer**

---

## 2. Averaging trees: bagging, random forests and extra trees

A deep tree has a small bias but a large variance. *Bagging* averages $B$ trees grown on bootstrap samples: the variance of the average of $B$ identically distributed predictors of variance $\sigma^2$ and pairwise correlation $\rho$ is $\rho\sigma^2 + (1-\rho)\sigma^2/B$. *Random forests* decrease $\rho$ by choosing each split among a random subset of $m$ features (`max_features`); *extra trees* also draw the thresholds at random.

**Question** *Predict before you run:* how does the variance of the predicted probabilities change from a single deep tree to the bagging of $100$ trees? Measure it: train both on $10$ random subsamples ($80\%$) of the training set, and compute, for each test loan, the variance of its predicted probability of `Risk` across the subsamples (averaged over the test loans). Compare also their test accuracies.

**Answer**

---

**Question** Random forests: for `max_features` in $\{1, 3, 8, 20, \text{all } p\}$ ($p$ = number of one-hot encoded features; all means bagging), compute the out-of-bag (OOB) accuracy of a forest of $300$ trees (`oob_score=True`). Which value is best? Compare with the recommendation $m \approx \sqrt{p}$.

**Answer**

---

**Question** Grow a random forest ($m = 8$) with `warm_start=True` from $10$ to $500$ trees, and plot, as a function of the number of trees, its OOB accuracy, its $5$-fold CV accuracy (for a few numbers of trees) and its test accuracy. Is the OOB estimate reliable? What does it cost?

**Answer**

---

**Question** Compare the impurity-based importances of the forest (`feature_importances_`) with the permutation importances on the test set (`permutation_importance`, $5$ repeats). Which features matter? Why can the two rankings differ?

**Answer**

---

## 3. Boosting

Boosting builds a sequence of *weak* learners, each one focusing on the errors of the previous ones. AdaBoost (Lecture 3 and PC3) uses labels $y_i \in \{-1, 1\}$, weights $w_i$ starting from $1/n$, and at each step $t$:
- fit a weak learner $h_t$ (here a *stump*, a tree of depth $1$) with the weights $w$, and compute its weighted error $\varepsilon_t = \sum_i w_i \mathbf{1}_{h_t(x_i) \neq y_i}$,
- set $\alpha_t = \frac{1}{2}\log\frac{1-\varepsilon_t}{\varepsilon_t}$,
- update $w_i \leftarrow w_i \exp(-\alpha_t y_i h_t(x_i))$ and normalize the weights,

and predicts $\text{sign}\left(\sum_t \alpha_t h_t(x)\right)$.

**Question** Complete the function `adaboost` below, run it for $200$ steps, and plot its training and test accuracies as functions of the number of steps. Compare with `AdaBoostClassifier(DecisionTreeClassifier(max_depth=1), n_estimators=200)` (its `staged_score`). Note: `scikit-learn`'s `estimator_weights_` are equal to $2\alpha_t$ (another normalization of the same algorithm); compare them with your $\alpha_t$.

In [ ]:
def adaboost(X, y, T):
  """AdaBoost with stumps; y in {-1, 1}. Returns the list of (alpha_t, h_t)."""
  n = len(y)
  w = np.full(n, 1 / n)
  learners = []
  for t in range(T):
    h = DecisionTreeClassifier(max_depth=1).fit(X, y, sample_weight=w)
    # complete: weighted error, alpha, update and normalization of the weights
    ...
  return learners

**Answer**

---

*Gradient boosting* generalizes boosting to any differentiable loss: each new tree fits the negative gradient of the loss at the current predictions, and is added with a small *learning rate* (shrinkage). The learning rate and the number of trees must be tuned together; *early stopping* stops adding trees when a validation score does not improve anymore.

**Question** Fit `GradientBoostingClassifier` (trees of depth $3$, $600$ trees) with learning rates $1$, $0.1$ and $0.01$, and plot the validation log loss as a function of the number of trees (use `staged_predict_proba` on a validation split of the training set). Then fit a `HistGradientBoostingClassifier` with early stopping: how many iterations does it keep?

**Answer**

---

## 4. Support vector machines

An SVM minimizes the hinge loss with a ridge penalty, $\frac{1}{n}\sum_i \max(0, 1 - y_i f(x_i)) + \lambda\|f\|^2$, with $C \propto 1/(n\lambda)$ (PC2). With a kernel $k(x, x') = \exp(-\gamma\|x - x'\|^2)$, $f$ lives in the space of functions $\sum_i \beta_i k(x_i, \cdot)$, and only the *support vectors* (the observations on or inside the margin) have $\beta_i \neq 0$. The features must be standardized.

**Question** Tune an RBF SVM (`SVC`) on the grid $C \in \{0.1, 1, 10, 100\}$, $\gamma \in \{0.001, 0.01, 0.1, 1\}$ with a $3$-fold CV of the accuracy. Plot the CV accuracy on the grid, and the number of support vectors as a function of $C$ (for the best $\gamma$). Comment.

**Answer**

---

## 5. An honest comparison with the cost-based gain

We finally compare the methods with the criterion that matters for the bank: the real-world gain of the Bayes decision computed from their probabilities (`cost_scorer_from_proba`, Lab 1). All the models are evaluated on the same $5 \times 2$ repeated stratified folds, so that their differences can be paired.

**Question** Evaluate the logistic regression, the pruned tree, the random forest, the extra trees, AdaBoost, the histogram gradient boosting and the SVM (its probabilities are obtained by a Platt calibration, `CalibratedClassifierCV(SVC(...), ensemble=False)`) on `RepeatedStratifiedKFold(n_splits=5, n_repeats=2)`. Display their mean gains with an interval of $\pm 2$ standard errors, compare the two best ones with the corrected resampled $t$-test of Lab 1, and evaluate the best one once on the test set.

**Answer**

---

## About this lab

**Use of AI.** Artificial intelligence tools have been used to revise the content of this lab. The main contributor takes full responsibility for this content.

**License.** The texts of this lab are under the [Creative Commons Attribution-ShareAlike 4.0](https://creativecommons.org/licenses/by-sa/4.0/) license (CC BY-SA 4.0), and its code under the [MIT](https://opensource.org/license/mit) license; the datasets keep their own licenses. Main contributor: E. Le Pennec. Contributors: S. Boucheron, A. Dieuleveut, A.K. Fermin, S. Gadat, S. Gaiffas, B. Goujaud, A. Guilloux, Ch. Keribin, Th. Klein, E. Matzner, J. Read, M. Sangnier, E. Scornet.